# Test 1 — Configuración del scope multitrigger

Verificación write/read de cada registro escribible del cfg vía `/dev/mem`.
Cubre el mapa clásico + los registros nuevos del multitrigger:
0x210 (shield), 0x214/0x218/0x21C (debug readbacks), 0x240/0x244 (máscara OR 32b por canal).

Base física asumida: `0x40100000`. Si tu device tree mapea distinto, ajustar `SCOPE_PHYS`.

Requiere correr como root (acceso a `/dev/mem`).

In [ ]:
import mmap, os, struct, random
from collections import OrderedDict

SCOPE_PHYS = 0x4010_0000
SCOPE_SIZE = 0x30000

fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
scope = mmap.mmap(fd, SCOPE_SIZE, mmap.MAP_SHARED,
                  mmap.PROT_READ | mmap.PROT_WRITE, offset=SCOPE_PHYS)

def w32(off, v):
    scope[off:off+4] = struct.pack('<I', v & 0xFFFFFFFF)

def r32(off):
    return struct.unpack('<I', scope[off:off+4])[0]

print('mmap OK at', hex(SCOPE_PHYS))

## Barrido principal — un puñado de valores por registro

Para registros *stored* (RW): write → read → `assert (read & mask) == (write & mask)`.
Para *pulse* (arm/rst/trig_sw/trig_dis_clr/new_trg_src) no hay readback estable; se omiten
del barrido write/read y se verifican aparte por efecto secundario.

In [ ]:
# (offset, mask, name, values_to_test)
TESTS = [
    # Threshold ch0/ch1 (14b signed)
    (0x08, 0x3FFF, 'set_tresh_ch0', [0, 1000, 0x1FFF, (-1000) & 0x3FFF, (-0x2000) & 0x3FFF]),
    (0x0C, 0x3FFF, 'set_tresh_ch1', [0, 1000, 0x1FFF, (-1000) & 0x3FFF, (-0x2000) & 0x3FFF]),
    # Delay ch0/ch1 (32b)
    (0x10,  0xFFFFFFFF, 'set_dly_ch0', [0, 1, 100, 16384, 0xFFFFFFFF]),
    (0x110, 0xFFFFFFFF, 'set_dly_ch1', [0, 1, 100, 16384, 0xFFFFFFFF]),
    # Decimation ch0/ch1 (17b, legal 1/8/64/1024/8192/65536)
    (0x14,  0x1FFFF, 'set_dec_ch0', [1, 8, 64, 1024, 8192, 65536]),
    (0x114, 0x1FFFF, 'set_dec_ch1', [1, 8, 64, 1024, 8192, 65536]),
    # Hysteresis ch0/ch1 (14b)
    (0x20, 0x3FFF, 'set_hyst_ch0', [20, 100, 0x1FFF]),
    (0x24, 0x3FFF, 'set_hyst_ch1', [20, 100, 0x1FFF]),
    # Filter coefs ch0 (18b aa, 25b bb/kk/pp)
    (0x30, 0x3FFFF,   'filt_aa_ch0', [0, 1, 0x3FFFF]),
    (0x34, 0x1FFFFFF, 'filt_bb_ch0', [0, 1, 0x1FFFFFF]),
    (0x38, 0x1FFFFFF, 'filt_kk_ch0', [0, 1, 0x1FFFFFF]),
    (0x3C, 0x1FFFFFF, 'filt_pp_ch0', [0, 1, 0x1FFFFFF]),
    (0x40, 0x3FFFF,   'filt_aa_ch1', [0, 1, 0x3FFFF]),
    (0x44, 0x1FFFFFF, 'filt_bb_ch1', [0, 1, 0x1FFFFFF]),
    (0x48, 0x1FFFFFF, 'filt_kk_ch1', [0, 1, 0x1FFFFFF]),
    (0x4C, 0x1FFFFFF, 'filt_pp_ch1', [0, 1, 0x1FFFFFF]),
    # AXI start/stop/dly ch0
    (0x50, 0xFFFFFFFF, 'axi_start_ch0', [0, 0x10000000, 0x40000000]),
    (0x54, 0xFFFFFFFF, 'axi_stop_ch0',  [0, 0x10001000, 0x40001000]),
    (0x58, 0xFFFFFFFF, 'axi_dly_ch0',   [0, 1, 0xFFFFFFFF]),
    # AXI start/stop/dly ch1
    (0x70, 0xFFFFFFFF, 'axi_start_ch1', [0, 0x10000000, 0x40000000]),
    (0x74, 0xFFFFFFFF, 'axi_stop_ch1',  [0, 0x10001000, 0x40001000]),
    (0x78, 0xFFFFFFFF, 'axi_dly_ch1',   [0, 1, 0xFFFFFFFF]),
    # Debouncer (20b global)
    (0x90, 0xFFFFF, 'set_deb_len', [1, 1000, 62500, 0xFFFFF]),
    # Filter bypass (4b)
    (0x98, 0xF, 'set_filt_byp', [0x0, 0x5, 0xF]),
    # Calib offset/gain ch0/ch1
    (0x200, 0x3FFF, 'calib_off_ch0',  [0, 1000, (-1000) & 0x3FFF]),
    (0x204, 0xFFFF, 'calib_gain_ch0', [0x8000, 0x4000, 0xC000]),
    (0x208, 0x3FFF, 'calib_off_ch1',  [0, 1000, (-1000) & 0x3FFF]),
    (0x20C, 0xFFFF, 'calib_gain_ch1', [0x8000, 0x4000, 0xC000]),
    # === multitrigger nuevos ===
    # Shield: {dur[31:16], 4'h0, dst[11:8], 4'h0, src[3:0]}
    (0x210, 0xFFFF0F0F, 'shield_cfg',
        [0x0000_0000, 0x00AA_0F0F, 0xFFFF_0F0F, 0x1234_0506]),
    # OR_MASK por canal (32b). Lectura devuelve set_trig_src latched.
    # Si el sistema arma+dispara, se limpia internamente; testear con scope no armado.
    (0x240, 0xFFFFFFFF, 'trg_src_act_ch0', [0x0001_FFFF, 0xFFFF_FFFF, 0x0000_0001, 0x0000_0000]),
    (0x244, 0xFFFFFFFF, 'trg_src_act_ch1', [0x0001_FFFF, 0xFFFF_FFFF, 0x0000_0001, 0x0000_0000]),
]

len(TESTS), 'registros'

In [ ]:
results = []
for off, mask, name, values in TESTS:
    for v in values:
        w32(off, v)
        rb = r32(off) & mask
        ex = v & mask
        results.append((name, off, v, rb, mask, rb == ex))

fails = [r for r in results if not r[5]]
print(f'Total: {len(results)}   OK: {len(results)-len(fails)}   FAIL: {len(fails)}')
for name, off, v, rb, mask, ok in fails:
    print(f'  FAIL {name} @{off:#06x}: wrote {v:#010x} read {rb:#010x} (mask {mask:#010x})')

In [ ]:
# Resumen por registro (agrupa)
from collections import defaultdict
per_name = defaultdict(lambda: [0, 0])  # [ok, total]
for name, off, v, rb, mask, ok in results:
    per_name[name][1] += 1
    per_name[name][0] += int(ok)

print(f'{"name":24s} {"off":>6s}  ok/total')
print('-' * 50)
for name in per_name:
    off = next(o for n, o, *_ in [(r[0], r[1]) for r in results] if n == name)
    ok, tot = per_name[name]
    flag = '' if ok == tot else '  <- FAIL'
    print(f'{name:24s} {off:#06x}  {ok}/{tot}{flag}')

## Readbacks de debug (0x214/0x218/0x21C)

Solo lectura. En idle (sin trigger) esperamos `shield_active=0`, `cnt=0`,
`snapshot=0`, `dis_act=0` (a menos que el sistema esté armado/disparado).

In [ ]:
print(f'shield  @0x214 = {r32(0x214):#010x}   (esperado 0x0 en idle)')
print(f'snapshot@0x218 = {r32(0x218):#010x}   (esperado 0x0 en idle)')
print(f'dis_act @0x21C = {r32(0x21C):#010x}   (esperado 0x0 en idle)')
print(f'mask act ch0 @0x240 = {r32(0x240):#010x}   (último valor latched)')
print(f'mask act ch1 @0x244 = {r32(0x244):#010x}')

## Sweep secundario — rangos amplios por registro

In [ ]:
def sweep(off, mask, name, values):
    fails = []
    for v in values:
        w32(off, v)
        rb = r32(off) & mask
        if rb != (v & mask):
            fails.append((v, rb))
    print(f'{name:24s} @{off:#06x}  {len(values)-len(fails)}/{len(values)} OK')
    for v, rb in fails[:5]:
        print(f'   FAIL wrote {v:#010x} read {rb:#010x}')

# Threshold barrido (14b signed wrap)
sweep(0x08, 0x3FFF, 'sweep_tresh_ch0', [v & 0x3FFF for v in range(-8192, 8192, 512)])
sweep(0x0C, 0x3FFF, 'sweep_tresh_ch1', [v & 0x3FFF for v in range(-8192, 8192, 512)])
# Hysteresis
sweep(0x20, 0x3FFF, 'sweep_hyst_ch0', list(range(0, 0x2000, 256)))
sweep(0x24, 0x3FFF, 'sweep_hyst_ch1', list(range(0, 0x2000, 256)))
# Delay
sweep(0x10,  0xFFFFFFFF, 'sweep_dly_ch0',  [0, 1] + [1 << i for i in range(0, 32, 4)] + [0xFFFFFFFF])
sweep(0x110, 0xFFFFFFFF, 'sweep_dly_ch1',  [0, 1] + [1 << i for i in range(0, 32, 4)] + [0xFFFFFFFF])
# Debouncer
sweep(0x90, 0xFFFFF, 'sweep_deb_len', [1, 10, 100, 1000, 62500, 0xFFFFF])
# Shield duration (parte alta de 0x210)
sweep(0x210, 0xFFFF0F0F, 'sweep_shield_dur',
      [(d << 16) | 0x0F0F for d in [0, 1, 100, 1000, 0xFFFF]])
# OR_MASK random + casos
random.seed(0)
vals = [0x0, 0x1, 0x10000, 0x0001_FFFF, 0xFFFF_FFFF] + [random.randint(0, 0xFFFFFFFF) for _ in range(27)]
sweep(0x240, 0xFFFFFFFF, 'sweep_trg_src_ch0', vals)
sweep(0x244, 0xFFFFFFFF, 'sweep_trg_src_ch1', vals)

## Pulsos (efecto secundario, no readback directo)

- `0x00 bit0` → `adc_arm_do[GV]`: arma.
- `0x00 bit1` → `adc_rst_do[GV]`: resetea.
- `0x04 byte/canal == 0x01` → `adc_trig_sw[GV]`.
- `0x94 bit0` → `trig_dis_clr[GV]`.
- Write a `0x240+4·GV` → pulso `new_trg_src[GV]` que engancha la máscara.

Verificamos efecto leyendo el `adc_state` @0x00 (cambia con arm/rst) y la
máscara activa @0x240 (engancha tras escribir).

In [ ]:
# Reset y comprobar estado
w32(0x00, 0x00000002)  # bit1 (rst) en byte 0 (canal 0)
s_before = r32(0x00)
print(f'adc_state tras rst: {s_before:#010x}')

# Engancha OR_MASK 0xAAAA_5555 en ch0 y verifica readback
w32(0x240, 0x0001_FFFF)
act0 = r32(0x240)
print(f'mask activa ch0 = {act0:#010x}   (esperado 0x0001FFFF)')
assert act0 == 0x0001_FFFF, 'new_trg_src no enganchó la máscara'

# Limpia
w32(0x240, 0x0000_0000)
w32(0x244, 0x0000_0000)
print('OK pulsos básicos')

In [ ]:
scope.close()
os.close(fd)
print('cerrado')